# Agent & Runner 連携テスト

このノートブックでは、Geminiが生成したと想定される分析コードを、`AnalysisRunner` を使って実際にローカルのParquetデータに対して実行できるかテストします。

In [10]:
import sys
from pathlib import Path
import pandas as pd

# プロジェクトルートをパスに追加
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.analyzer.runner import AnalysisRunner
from dotenv import load_dotenv

load_dotenv(project_root / ".env")

runner = AnalysisRunner(data_dir=str(project_root / "data"))

## 1. データの読み込み
すでに保存されているParquetデータを読み込みます。以前のテストで保存した日付を指定してください。

In [3]:
# テストデータとして1日分読み込む
target_date = "2024-05-15" # 取得済みの正しい日付に変更してください
df = runner.load_daily_bars(target_date, target_date)

print(f"Loaded {len(df)} records.")
df.head()

INFO:src.analyzer.runner:Loading 1 files...


Loaded 4359 records.


,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo
0,2024-05-15,13010,3760.0,3770.0,3725.0,3740.0,0,0,23600.0,8.833750e+07,1.0,3760.0,3770.0,3725.0,3740.0,23600.0
1,2024-05-15,13050,2932.0,2947.5,2916.0,2917.5,0,0,549760.0,1.608675e+09,1.0,2932.0,2947.5,2916.0,2917.5,549760.0
2,2024-05-15,13060,2899.0,2915.0,2885.0,2887.5,0,0,1036310.0,3.002599e+09,1.0,289.9,291.5,288.5,288.8,10363100.0
3,2024-05-15,13080,2865.0,2882.0,2853.0,2855.0,0,0,205241.0,5.880859e+08,1.0,2865.0,2882.0,2853.0,2855.0,205241.0
4,2024-05-15,13090,42350.0,42500.0,42000.0,42500.0,0,0,204.0,8.634210e+06,1.0,42350.0,42500.0,42000.0,42500.0,204.0


## 2. Gemini生成コードのシミュレーション
Geminiが `src/agent/schema.py` の形式で出力したと想定されるコードを準備します。

In [8]:
# 例：ボラティリティ（標準偏差）を計算して、異常に動きが大きい銘柄を抽出するコード
ai_code = """
# 始値と終値の乖離率を計算
df['diff_pct'] = (df['C'] - df['O']) / df['O'] * 100

# 乖離率の絶対値が5%以上の銘柄を抽出
output_df = df[df['diff_pct'].abs() > 5.0].sort_values('diff_pct', ascending=False)

result = f"合計 {len(output_df)} 銘柄が5%以上の値動きを見せました。"
"""

print("Simulation Code:")
print(ai_code)

Simulation Code:

# 始値と終値の乖離率を計算
df['diff_pct'] = (df['C'] - df['O']) / df['O'] * 100

# 乖離率の絶対値が5%以上の銘柄を抽出
output_df = df[df['diff_pct'].abs() > 5.0].sort_values('diff_pct', ascending=False)

result = f"合計 {len(output_df)} 銘柄が5%以上の値動きを見せました。"



## 3. Runnerでの実行
実際に `run_code` を呼び出して、結果を確認します。

In [9]:
res = runner.run_code(df, ai_code)

if res["success"]:
    print("✅ 実行成功")
    print(f"Message: {res['result']}")
    display(res["output_df"]) # 抽出されたデータフレームを表示
else:
    print("❌ 実行失敗")
    print(f"Error: {res['error']}")

✅ 実行成功
Message: 合計 235 銘柄が5%以上の値動きを見せました。


,Date,Code,O,H,L,C,UL,LL,Vo,Va,AdjFactor,AdjO,AdjH,AdjL,AdjC,AdjVo,diff_pct
2102,2024-05-15,52160,252.0,332.0,248.0,328.0,1,0,16841600.0,5.105079e+09,1.0,252.0,332.0,248.0,328.0,16841600.0,30.158730
2389,2024-05-15,60260,7130.0,8680.0,7130.0,8680.0,1,0,43200.0,3.617240e+08,1.0,7130.0,8680.0,7130.0,8680.0,43200.0,21.739130
3357,2024-05-15,76890,716.0,851.0,686.0,851.0,1,0,1403000.0,1.131961e+09,1.0,716.0,851.0,686.0,851.0,1403000.0,18.854749
812,2024-05-15,28130,3715.0,4415.0,3570.0,4415.0,1,0,26100.0,1.012765e+08,1.0,3715.0,4415.0,3570.0,4415.0,26100.0,18.842530
2578,2024-05-15,63300,885.0,1034.0,865.0,1034.0,1,0,1822400.0,1.771948e+09,1.0,885.0,1034.0,865.0,1034.0,1822400.0,16.836158
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2450,2024-05-15,61310,1630.0,1630.0,1357.0,1375.0,0,0,117700.0,1.747660e+08,1.0,1630.0,1630.0,1357.0,1375.0,117700.0,-15.644172
3279,2024-05-15,75250,4340.0,4340.0,3650.0,3660.0,0,1,46000.0,1.790270e+08,1.0,4340.0,4340.0,3650.0,3660.0,46000.0,-15.668203
1976,2024-05-15,48940,3570.0,3570.0,3000.0,3010.0,0,0,413000.0,1.342098e+09,1.0,3570.0,3570.0,3000.0,3010.0,413000.0,-15.686275
3258,2024-05-15,74940,394.0,394.0,325.0,332.0,0,0,1021300.0,3.539395e+08,1.0,394.0,394.0,325.0,332.0,1021300.0,-15.736041
